## Unified Risk Engine

#### Libraries to be imported

In [1]:
# Libraries to be imported for the project
import os
import sys
import sqlite3
import warnings
from typing import List, Tuple, Dict, Union, Optional
from datetime import datetime, timedelta

# Core scientific and numerical libraries
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import norm, chi2

# financial  data acquisition and analysis libraries
import yfinance as yf

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Runtime settings
warnings.filterwarnings("ignore")
np.random.seed(42) # ensure deterministic results for reproducibility
pd.set_option("display.precision", 4) # set pandas display precision for better readability
pd.set_option("display.max_columns", 15) # display all columns in pandas DataFrames
plt.style.use('seaborn-v0_8-whitegrid') # set default plot style for matplotlib

# Environment verification
print("Core Quantity Libraries Loaded Successfully.")


Core Quantity Libraries Loaded Successfully.


1. Market Data Ingestion

Pulls 500 trading days of historical market data for 5 risk benchmarks:
- `BZ=F`: ICE Brent Crude Oil
- `CL=F`: NYMEX WTI Crude Oil
- `NG=F`: NYMEX Henry Hub Natural Gas
- `^OVX`: CBOE Crude Oil Volatility Index
- `^TNX`: US 10-Year Treasury Yield

In [10]:
tickers = {
    "BZ=F": "Brent",
    "CL=F": "WTI",
    "NG=F": "NatGas",
    "^OVX": "OVX",
    "^TNX": "US10Y"
}

start_date = "2007-01-01"

df = yf.download(list(tickers.keys()), start=start_date, progress=False, auto_adjust=False)['Adj Close']

df = df.rename(columns=tickers)
df.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 4998 entries, 2007-01-02 to 2026-11-06
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Brent   4799 non-null   float64
 1   WTI     4995 non-null   float64
 2   NatGas  4996 non-null   float64
 3   OVX     4906 non-null   float64
 4   US10Y   4992 non-null   float64
dtypes: float64(5)
memory usage: 234.3 KB


2. Calendar Synchronization and Return Calculation
Using Forward filling upto 3 days to map missing values
Returns calculated are Log returns

In [11]:
df_prices = df.ffill(limit=3).dropna()
df_returns = np.log(df_prices / df_prices.shift(1)).dropna()

print(f"Clean prices shape : {df_prices.shape}, Log returns shape : {df_returns.shape}")
df_returns.tail(5)

Clean prices shape : (4845, 5), Log returns shape : (4842, 5)


Ticker,Brent,WTI,NatGas,OVX,US10Y
Date,,,,,
2026-11-02,-0.1159,-0.0764,-0.0146,0.0695,-0.0071
2026-11-03,-0.0112,-0.0038,0.0136,-0.0161,0.0118
2026-11-04,-0.0907,-0.0866,0.0219,-0.0478,-0.0118
2026-11-05,0.0381,0.0269,-0.0041,0.0249,-0.0117
2026-11-06,-0.0095,-0.0023,0.0064,-0.0196,0.0076
